## Neural Network Implementations Development 
This purpose this notebook is to incrementally implement a neural network to strengthen understanding of the algorithm

In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns 

from typing import Tuple

## Forward Pass for 1 Hidden Layer
Through making use of matrix operation of weights, bias and activation through numpy the model is able to make prediction. Also For simplicity in batch operations, I chose row-vector inputs (samples × features), which is equivalent to the column-vector convention seen in textbooks.

In [11]:
def enforce_row_vector(x: np.ndarray) -> np.ndarray:
    """Force input into shape (1, n_features) if the input is 1 dimensional."""
    x = np.array(x, dtype=float)        # ensure NumPy array
    if x.ndim == 1: 
        return x.reshape(1, -1 ) # row vector
    else:
        return x 

In [2]:
def forward_pass_1HL(feature_array: np.ndarray,hidden_weights: np.ndarray, 
                     hidden_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_vector: np.ndarray)-> Tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    '''
        Conducts the forward pass of a 1 hidden layer neural net using the ReLu activation function
        and evaluates the results using MSE
        
        Args:
            feature_array (np.ndarray): Inputted feature array where each col represents a feature and each row a sample 
            hidden_weights (np.ndarray): Weight matrix for the hidden layer
            hidden_bias (np.ndarray): Bias vector for the hidden layer
            out_weights (np.ndarray) Weight matrix for the output layer
            out_bias (np.ndarray): Bias vector for the output layer
            target_vector (np.ndarray): Target vector for preformance evaluation. Where each col represents a target and each row a sample.   
        Return:
            np.ndarray: Output activations of the network
            np.ndarray: The difference vector of the output activations and the true y
            float: The mean squared error of the network's predictions
            np.ndarray: The neuron activation values of the last hidden layer
    '''
    feature_array = enforce_row_vector(feature_array)
    
    hidden_activations = np.maximum(feature_array @ hidden_weights + hidden_bias, 0)
    output_activations = hidden_activations @ out_weights + out_bias
    error = (target_vector - output_activations)
    half_mse = (1/2) * np.mean((error) ** 2)
    return output_activations, error, half_mse, hidden_activations 
    

## Computing Gradients 
Through using a matrix algebra implementaions of weights and biases gradient formulas obtained from back propogation we can get the gradient vectors of each layer. Also since the equations are implemented through matrix algebra we can use numpy vector operations for efficiency, avoiding the the cost of using loops.   

In [3]:
def compute_output_gradients(error: np.ndarray, hidden_activations:np.ndarray) -> np.ndarray:
    '''
    Computes the gradients of the weights and biases in the output layer

    Args: 
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden_activations (np.ndarray): The neuron activation values of the last hidden layer

    Returns:
        np.ndarray: Gradient array of the output units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the output units containing the gradient values of the weights
        
    '''
 
    bias_gradients = np.ones((1, feature_array.shape[1])) @ (-1 * error )
    output_gradient_matrix = hidden_activations @ (-1 * error)
    return bias_gradients, weight_gradients 

In [4]:
def compute_last_hidden_gradients(error: np.ndarray, hidden_activations:np.ndarray, 
                             out_weights: np.ndarray, feature_array: np.ndarray) -> np.ndarray:
    '''
    Computes the gradients of the weights and biases in the hidden layer.

    Args: 
        feature_array (np.ndarray): Inputted feature array
        error (np.ndarray): The difference vector of the output activations and the true y 
        hidden_activations (np.ndarray): The neuron activation values of the last hidden layer
        out_weights (np.ndarray): The weight matrix for the output neurons 

    Returns:
        np.ndarray: Gradient array of the hidden units containing the gradient values of the biases
        np.ndarray: Gradient matrix of the hidden units containing the gradient values of the weights
        
    
    '''
    relu_mask = hidden_activations > 0
    relu_derivative = relu_mask.astype(int)
    bias_gradients = np.ones((1, feature_array.shape[1])) @ (-1 * error @ out_weights.T ) * relu_derivative
    weight_gradients = feature_array @ (-1 * error @ out_weights.T )) * relu_derivative
    return bias_gradients, weight_gradients

## Back Propagation 
Since the forward pass function allows for batch inputs, the loss of the whole network can be calculated by inputting the full training input into the forward pass function. When the outputs of the forward pass is inputted into the gradient functions the resulting output is essentially a summation of the gradient values of the weights and biases for each training example. Thus to get the mean gradients values you simply have to divide the arrays by 1/n, where n is the sample size.   

In [5]:
def back_propagation(training_feature_array: np.ndarray,hidden_weights: np.ndarray, 
                     hidden_bias:np.ndarray, out_weights: np.ndarray, out_bias:np.ndarray,
                    target_matrix: np.ndarray)-> Tuple[np.ndarray, np.ndarray]:
    '''
    Runs  backpropogation on the training data to generate the mean gradients of the weights and biases for 
    a 1 hidden layer neural network.

    Args:
        training_feature_array (np.ndarray): Inputted training feature array.  Each row represent xi
        hidden_weights (np.ndarray): Weight matrix for the hidden layer
        hidden_bias (np.ndarray): Bias vector for the hidden layer
        out_weights (np.ndarray) Weight matrix for the output layer
        out_bias (np.ndarray): Bias vector for the output layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Each row represent yi
    Return:
        np.ndarray: The mean output layer bias gradients 
        np.ndarray: The mean output layer weight gradients 
        np.ndarray: The mean hidden layer bias gradients 
        np.ndarray: The mean hidden layer weight gradients 
    '''
 
    
    output_activations, error, half_mse, hidden_activations = forward_pass_1HL(training_feature_array, hidden_weights, 
                         hidden_bias, out_weights, out_bias,
                        target_matrix)
    
    output_bias_gradients, output_weight_gradients = compute_output_gradients(error, hidden_activations, 
                             out_weights, feature_array)
        
    hidden_bias_gradients, hidden_weight_gradients = compute_last_hidden_gradients(error, hidden_activations, 
                                 out_weights, feature_array)

    n = len(training_feature_array)
    mean_out_weight_gradients = (1/n) * output_weight_gradients
    mean_out_bias_gradients = (1/n) * output_bias_gradients
    mean_hidden_weight_gradients = (1/n) * hidden_weight_gradients
    mean_hidden_bias_gradients = (1/n) * hidden_bias_gradients

    return mean_out_bias_gradients, mean_out_weight_gradients, mean_hidden_bias_gradients, mean_hidden_weight_gradients
    

## Gradient Updating
Once we have the gradient vectors and matrices we can update the weight and biase values through the regular convention using the chosen learning rate. 

In [6]:
def gradient_descent_update(learning_rate: float, old_out_weights:np.ndarray, 
                            old_out_bias: np.ndarray, output_bias_gradients: np.ndarray, 
                            output_weight_gradients: np.ndarray, old_hidden_weights: np.ndarray, 
                            old_hidden_bias: np.ndarray, hidden_bias_gradients: np.ndarray, 
                            hidden_weight_gradients: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent updating of the weights and biases of a 1 hidden layer neural network

    Args: 
        learning_rate (float): The learning rate for the neural network 
        old_out_weights (np.ndarray): The current weights of the output layer  
        old_out_bias(np.ndarray): The current biases of the output layer
        output_bias_gradients (np.ndarray): The gradient values of the current output layer biases
        output_weights_gradients (np.ndarray): The gradient values of the current output layer weights
        old_hidden_weights(np.ndarray): The current weights of the hidden layer 
        old_hidden_bias (np.ndarray): The current biases of the hidden layer 
        hidden_bias_gradients (np.ndarray):  The gradient values of the current hidden layer biases
        hidden_bias_gradients (np.ndarray): The gradient values of the current hidden layer weights
    
    Return:
        np.ndarray: Updated output layer weights
        np.ndarray: Updated output layer biases
        np.ndarray: Updated hidden layer weights
        np.ndarray: Updated hidden layer biases

    '''

    updated_output_weights = old_out_weights - learning_rate * output_weight_gradients
    updated_output_biases = old_out_bias - learning_rate * output_bias_gradients
    updated_hidden_weights = old_hidden_weights - learning_rate * hidden_weight_gradients
    updated_hidden_biases = old_hidden_bias - learning_rate * hidden_weight_gradients

    return updated_output_weights, updated_output_biases, updated_hidden_weights, updated_hidden_biases

## 1 Hidden Layer Nerual Net with early stopping validation and HE initialisation 
To deal with weight initailisation, I went with He as it would work well with the ReLu activation function used through the network. In future iterations, other initialisation options will be added along with other activation functions. In regards to the stopping mechanisms, “I used early stopping on validation loss with a patience of k epochs. This prevents overfitting while also providing a clear stopping mechanism. This I felt was the most practical option, however, mechanisms like gradient norm threshold do deserve experimentation even if they are more unstable, so they will be added in the next stage as an option when the nerual net is wrapped as a class.

In [10]:
def neural_net_1HL(training_feature_array: np.ndarray, hidden_neurons: int, 
                    target_matrix: np.ndarray, learning_rate: float, 
                   max_epochs: int, validation_features: np.ndarray, 
                   validations_target: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Conducts gradient descent on a 1 hidden layer neural network to output the trained weights for predictions

    Args:
        training_feature_array (np.ndarray): Inputted training feature array.  Each row represent xi
        hidden_neurons (int): Number of neuron in the hidden layer
        target_matrix (np.ndarray): Target matrix for preformance evaluation. Each row represent yi
        learning_rate (float): The learning rate for the neural network 
        max_epochs (int): Max number of iterations of gradient descent to conduct.
        validation_features (np.ndarray): Validation feature array to assess improvements in the neural net for stopping criteria. 
        validation_features (np.ndarray): Validation target array to assess improvements in the neural net for stopping criteria. 
        k (int): Number of epochs allowed without a change in validation loss before stopping gradient descent  
    Returns:
        np.ndarray: Trained output layer weights
        np.ndarray: Trained output layer biases
        np.ndarray: Trained hidden layer weights
        np.ndarray: Trained hidden layer biases
    '''
    hidden_weights = np.random.rand(training_feature_array.shape[1], hidden_neurons) * np.sqrt(2 / training_feature_array.shape[1])
    hidden_bias = np.zeros((1, hidden_neurons))
    out_weights =  np.random.rand(hidden_neurons, target_matrix.shape[1]) * np.sqrt(2 / hidden_neurons)
    out_bias = np.zeros((1, target_matrix.shape[1]))
    previous_validation_loss = 0
    epoch_completed = 0
    no_change_epochs = 0 
    epoch_count = 0 
    while not stop_descent:
        out_bias_gradients, out_weight_gradients, hidden_bias_gradients, hidden_weight_gradients = back_propagation(training_feature_array ,
                                                                                                                   hidden_weights, hidden_bias, 
                                                                                                                   out_weights, out_bias,
                                                                                                                   target_matrix)
        
        output_weights, output_biases, hidden_weights, hidden_biases= gradient_descent_update(learning_rate, out_weights, 
                                                                                                out_bias, out_bias_gradients, 
                                                                                                out_weight_gradients, hidden_weights, 
                                                                                                hidden_bias, hidden_bias_gradients, 
                                                                                                hidden_weight_gradients)
        validation_loss = forward_pass_1HL(validation_features, hidden_weights, 
                     hidden_biases, out_weights, out_biases,
                    validation_target)[2]
        if validation_loss == previous_validation_loss:
            no_change_epochs += 1 
        else:
            no_change_epochs = 0
        
        previous_validation_loss = validation_loss

        if no_change_epochs >= k:
            stop_descent = True

        epoch_count += 1 
        if epoch_count >= max_epochs:
            stop_descent = True 
            

        
    return output_weights, output_biases, hidden_weights, hidden_biases 
        